# Final Architecture Results — Phase 4 & Best Models

**Focus:** Phase 4 hybrid architecture outcomes + overall best models across all research phases.

Condensed analysis notebook (complementary to `results_analysis.ipynb`).

## 1. Setup & Data Loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import wandb

# Sync W&B offline runs (optional)
print("Syncing W&B offline runs...")
import subprocess
try:
    subprocess.run(["wandb", "sync", "--sync-all"], capture_output=True, timeout=30)
    print("✓ W&B sync complete")
except Exception as e:
    print(f"⚠ W&B sync skipped: {e}")

project_root = Path.cwd().parent

PHASE4_DIR = project_root / "results" / "final_architecture_phase4"
ALL_PHASES_DIRS = {
    "Phase 1": project_root / "results" / "baselines_qat_phase1",
    "Phase 2": project_root / "results" / "alexnet_qat_phase2",
    "Phase 3": project_root / "results" / "compensation_phase3",
    "Phase 4": PHASE4_DIR,
}

FIGURES_DIR = project_root / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 11})
print("✓ Setup complete")

In [ ]:
# Load Phase 4 comparison table
phase4_comp = pd.read_csv(PHASE4_DIR / "final_comparison.csv")
phase4_comp["phase"] = "Phase 4"

# Load all cross-phase comparisons
all_comps = [phase4_comp]
for phase_name, phase_dir in list(ALL_PHASES_DIRS.items())[:-1]:  # Skip Phase 4, already loaded
    csv_path = phase_dir / "final_comparison.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df["phase"] = phase_name
        all_comps.append(df)

df_all = pd.concat(all_comps, ignore_index=True)

# Load per-model summaries (Phase 4 only for now)
phase4_summaries = {}
for json_file in sorted(PHASE4_DIR.glob("*_summary.json")):
    if json_file.name != "experiment_summary.json":
        with open(json_file) as f:
            phase4_summaries[json_file.stem] = json.load(f)

print(f"✓ Loaded Phase 4: {len(phase4_comp)} models")
print(f"✓ Loaded all phases: {len(df_all)} model variants")
print(f"✓ Loaded Phase 4 summaries: {len(phase4_summaries)} models")

## 2. Phase 4 Summary Table

In [ ]:
print("Phase 4 — Final Architecture Models (FP32 & INT8 Quantization)")
print("="*80)

phase4_display = phase4_comp[["model", "precision", "top1_%", "top5_%", "loss", "params_M", "size_MB"]].copy()
phase4_display = phase4_display.sort_values(["precision", "top1_%"], ascending=[False, False])

display(phase4_display.reset_index(drop=True))

print("\n📊 Phase 4 Best Models:")
fp32_best = phase4_comp[phase4_comp["precision"] == "FP32"].nlargest(1, "top1_%")
int8_best = phase4_comp[phase4_comp["precision"] == "INT8"].nlargest(1, "top1_%")

if not fp32_best.empty:
    row = fp32_best.iloc[0]
    print(f"  FP32 Best: {row['model']:40s} | {row['top1_%']:6.2f}% top-1 | {row['size_MB']:6.2f} MB")

if not int8_best.empty:
    row = int8_best.iloc[0]
    print(f"  INT8 Best: {row['model']:40s} | {row['top1_%']:6.2f}% top-1 | {row['size_MB']:6.2f} MB")

## 3. Phase 4 Visualizations

### 3.1 FP32 vs INT8 Quantization Impact

In [ ]:
# Prepare Phase 4 quantization comparison
_df = phase4_comp.copy()
_df["base_model"] = _df["model"].str.replace("_INT8", "", regex=False)

has_both = _df.groupby("base_model")["precision"].apply(
    lambda x: {"FP32", "INT8"}.issubset(set(x))
)
paired = has_both[has_both].index.tolist()

if paired:
    df_paired = _df[_df["base_model"].isin(paired)].copy()
    df_pivot = (
        df_paired
        .pivot_table(index="base_model", columns="precision", values="top1_%")
        .reset_index()
    )
    df_pivot["drop"] = df_pivot["FP32"] - df_pivot["INT8"]
    df_pivot = df_pivot.sort_values("FP32", ascending=False).reset_index(drop=True)

    x = np.arange(len(df_pivot))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, df_pivot["FP32"], w, label="FP32", color="steelblue", alpha=0.87)
    ax.bar(x + w/2, df_pivot["INT8"], w, label="INT8", color="darkorange", alpha=0.87)

    for i, row in df_pivot.iterrows():
        color = "red" if row["drop"] > 0.5 else "green" if row["drop"] < 0 else "gray"
        sign = "-" if row["drop"] > 0 else "+"
        ax.annotate(
            f"{sign}{abs(row['drop']):.2f}%",
            xy=(x[i] + w/2, row["INT8"]),
            xytext=(0, 5), textcoords="offset points",
            ha="center", fontsize=9, color=color, fontweight="bold",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(df_pivot["base_model"], rotation=30, ha="right")
    ax.set_ylabel("Top-1 Accuracy (%)")
    ax.set_title("Phase 4: FP32 vs INT8 Quantization Impact")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "phase4_fp32_vs_int8.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved phase4_fp32_vs_int8.png")

### 3.2 Phase 4 Model Efficiency

In [ ]:
# Efficiency: accuracy per MB — FP32 vs INT8 comparison
phase4_fp32 = phase4_comp[phase4_comp["precision"] == "FP32"].copy()
phase4_int8 = phase4_comp[phase4_comp["precision"] == "INT8"].copy()

phase4_fp32["efficiency"] = phase4_fp32["top1_%"] / phase4_fp32["size_MB"]
phase4_int8["efficiency"] = phase4_int8["top1_%"] / phase4_int8["size_MB"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# LEFT: Scatter plot — Size vs Accuracy (FP32 circles, INT8 squares)
for idx, row in phase4_fp32.iterrows():
    ax1.scatter(row["size_MB"], row["top1_%"], s=220, alpha=0.8, 
               edgecolors="black", lw=1.5, color="steelblue", zorder=3)
    model_label = row["model"].replace("alexnet_final_", "")
    ax1.annotate(model_label, 
                 (row["size_MB"], row["top1_%"]),
                 xytext=(6, 8), textcoords="offset points", fontsize=9, fontweight="bold")

for idx, row in phase4_int8.iterrows():
    ax1.scatter(row["size_MB"], row["top1_%"], s=220, alpha=0.6, marker="s",
               edgecolors="black", lw=1.5, color="darkorange", zorder=3)
    model_label = row["model"].replace("_INT8", "").replace("alexnet_final_", "")
    ax1.annotate(f"{model_label} (INT8)", 
                 (row["size_MB"], row["top1_%"]),
                 xytext=(6, -12), textcoords="offset points", fontsize=8, alpha=0.7)

# Add legend patch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=10, label="FP32"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="darkorange", markersize=10, label="INT8"),
]
ax1.legend(handles=legend_elements, loc="best", fontsize=10)
ax1.set_xlabel("Model Size (MB)")
ax1.set_ylabel("Top-1 Accuracy (%)")
ax1.set_title("Phase 4: Size vs Accuracy (FP32 ○ INT8 □)")
ax1.grid(True, alpha=0.3)

# RIGHT: Bar chart — Efficiency comparison
model_names = phase4_fp32["model"].str.replace("alexnet_final_", "").values
fp32_effs = phase4_fp32["efficiency"].values

# Match INT8 to FP32 for comparison
int8_effs = []
for fp32_model in phase4_fp32["model"]:
    int8_model = fp32_model + "_INT8"
    match = phase4_int8[phase4_int8["model"] == int8_model]
    if not match.empty:
        int8_effs.append(match.iloc[0]["efficiency"])
    else:
        int8_effs.append(0)

x = np.arange(len(model_names))
w = 0.35

ax2.bar(x - w/2, fp32_effs, w, label="FP32", color="steelblue", alpha=0.85, edgecolor="black", lw=1.2)
ax2.bar(x + w/2, int8_effs, w, label="INT8", color="darkorange", alpha=0.85, edgecolor="black", lw=1.2)

# Add value labels
for i, (fp32, int8) in enumerate(zip(fp32_effs, int8_effs)):
    ax2.text(i - w/2, fp32 + 0.15, f"{fp32:.1f}", ha="center", fontsize=8, fontweight="bold")
    if int8 > 0:
        ax2.text(i + w/2, int8 + 0.15, f"{int8:.1f}", ha="center", fontsize=8, fontweight="bold")

ax2.set_xticks(x)
ax2.set_xticklabels(model_names, rotation=20, ha="right", fontsize=10)
ax2.set_ylabel("Efficiency (Accuracy % per MB)")
ax2.set_title("Phase 4: Parameter Efficiency Comparison")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "phase4_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved phase4_efficiency.png (FP32 + INT8 comparison)")

### 3.3 Phase 4 Ablation Summary

In [ ]:
# Load Phase 4 ablation results
ablation_path = PHASE4_DIR / "ablation_results.csv"
if ablation_path.exists():
    ablation = pd.read_csv(ablation_path)
    
    print("\n📈 Phase 4 Ablation: Hybrid vs Single-Mechanism Parents")
    print("="*80)
    
    # Map hybrids to their parents for display
    parent_map = {
        "alexnet_final_bottleneck_fire": ["Bottleneck", "Fire"],
        "alexnet_final_fire_residual": ["Fire", "Residual"],
        "alexnet_final_bottleneck_residual": ["Bottleneck", "Residual"],
        "alexnet_final_depthwise_fire": ["Depthwise-Sep", "Fire"],
    }
    
    display_cols = ["hybrid", "hybrid_top1", "best_parent_top1", "delta_vs_best_parent"]
    ablation_display = ablation[[c for c in display_cols if c in ablation.columns]].copy()
    ablation_display.columns = ["Hybrid Model", "Hybrid Top-1 %", "Best Parent Top-1 %", "Delta %"]
    
    # Add parent info
    ablation_display["Parents"] = ablation["hybrid"].map(lambda x: " + ".join(parent_map.get(x, ["Unknown"])))
    
    display(ablation_display[["Hybrid Model", "Parents", "Hybrid Top-1 %", "Best Parent Top-1 %", "Delta %"]])
    
    # Visualization with parent labels
    fig, ax = plt.subplots(figsize=(12, 5))
    
    x = np.arange(len(ablation))
    w = 0.35
    
    hybrid_names = ablation["hybrid"].str.replace("alexnet_final_", "").values
    ax.bar(x - w/2, ablation["hybrid_top1"], w, label="Hybrid", color="#3498db", alpha=0.85, edgecolor="black", lw=1.2)
    ax.bar(x + w/2, ablation["best_parent_top1"], w, label="Best Parent", color="#95a5a6", alpha=0.85, edgecolor="black", lw=1.2)
    
    # Annotations: delta + parent info
    for i, (idx, row) in enumerate(ablation.iterrows()):
        delta = row["delta_vs_best_parent"]
        color = "green" if delta > 0 else "red"
        sign = "+" if delta > 0 else ""
        
        # Parent names
        parents = parent_map.get(row["hybrid"], ["Unknown"])
        parent_label = " + ".join(parents)
        
        # Delta annotation
        ax.text(x[i], max(row["hybrid_top1"], row["best_parent_top1"]) + 0.8,
                f"{sign}{delta:.1f}%", ha="center", fontsize=9, color=color, fontweight="bold")
        
        # Parent label below x-axis
        ax.text(x[i] + w/2, -2.5, f"({parent_label})", ha="center", fontsize=8, 
               style="italic", color="gray", rotation=0)
    
    ax.set_xticks(x)
    ax.set_xticklabels(hybrid_names, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("Top-1 Accuracy (%)")
    ax.set_title("Phase 4 Ablation: Hybrid Architectures vs Single-Mechanism Parents")
    ax.set_ylim(bottom=-3.5)
    ax.legend(fontsize=10, loc="upper right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "phase4_ablation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved phase4_ablation.png (with parent specifications)")
else:
    print("[skip] ablation_results.csv not found")

## 4. Cross-Phase Best Models

In [ ]:
# Top 10 models by FP32 accuracy across all phases
df_fp32_all = df_all[df_all["precision"] == "FP32"].copy()
top10 = df_fp32_all.nlargest(10, "top1_%")[["phase", "model", "top1_%", "top5_%", "params_M", "size_MB"]]

print("\n🏆 Top 10 Models — All Phases (FP32)")
print("="*80)
display(top10.reset_index(drop=True))

# Highlight Phase 4 in top 10
phase4_in_top10 = top10[top10["phase"] == "Phase 4"]
if not phase4_in_top10.empty:
    print(f"\n✓ Phase 4 models in top 10: {len(phase4_in_top10)}")
    for _, row in phase4_in_top10.iterrows():
        print(f"  • {row['model']:40s} | {row['top1_%']:6.2f}%")
else:
    print("\n⚠ No Phase 4 models in top 10 by accuracy")

In [ ]:
# Cross-phase efficiency frontier
df_fp32_all["efficiency"] = df_fp32_all["top1_%"] / df_fp32_all["size_MB"]

def pareto_front_mask(xs, ys):
    """Boolean mask: True if point is Pareto-optimal (minimize x, maximize y)."""
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    dominated = np.zeros(len(xs), dtype=bool)
    for i in range(len(xs)):
        for j in range(len(xs)):
            if i != j and xs[j] <= xs[i] and ys[j] >= ys[i] and (xs[j] < xs[i] or ys[j] > ys[i]):
                dominated[i] = True
                break
    return ~dominated

phase_colors = {"Phase 1": "#e74c3c", "Phase 2": "#f39c12", "Phase 3": "#9b59b6", "Phase 4": "#27ae60"}

fig, ax = plt.subplots(figsize=(13, 8))

# Plot all models by phase
for phase in ["Phase 1", "Phase 2", "Phase 3", "Phase 4"]:
    sub = df_fp32_all[df_fp32_all["phase"] == phase]
    if sub.empty:
        continue
    ax.scatter(sub["size_MB"], sub["top1_%"],
               c=phase_colors[phase], marker="o", s=100,
               label=phase, alpha=0.75, edgecolors="white", lw=0.5, zorder=3)

# Pareto front overlay
if len(df_fp32_all) >= 2:
    pf_mask = pareto_front_mask(df_fp32_all["size_MB"].values, df_fp32_all["top1_%"].values)
    pf = df_fp32_all[pf_mask].sort_values("size_MB")
    ax.step(pf["size_MB"], pf["top1_%"], where="post",
            color="black", lw=2, ls="--", alpha=0.6, label="Pareto front", zorder=2)

# Annotations for interesting models
for _, row in df_fp32_all.iterrows():
    if row["top1_%"] > 45 or (row["size_MB"] < 10 and row["top1_%"] > 40):
        label = row["model"].replace("alexnet_", "").replace("_", " ")[:20]
        ax.annotate(label, (row["size_MB"], row["top1_%"]),
                    xytext=(3, 3), textcoords="offset points", fontsize=7.5, alpha=0.8)

ax.set_xscale("log")
ax.set_xlabel("Model Size (MB, log scale)")
ax.set_ylabel("Top-1 Accuracy (%)")
ax.set_title("Efficiency Frontier: All Phases — Size vs Accuracy")
ax.legend(loc="best", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cross_phase_efficiency_frontier.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved cross_phase_efficiency_frontier.png")

## 5. Key Findings & Recommendations

In [ ]:
print("\n" + "="*80)
print("RESEARCH HIGHLIGHTS")
print("="*80)

# Best overall (all phases)
best_overall = df_fp32_all.nlargest(1, "top1_%").iloc[0]
print(f"\n🥇 Best Overall Accuracy (All Phases):")
print(f"   {best_overall['model']:40s} [{best_overall['phase']}]")
print(f"   Top-1: {best_overall['top1_%']:.2f}%  |  Params: {best_overall['params_M']:.2f}M  |  Size: {best_overall['size_MB']:.2f} MB")

# Best tiny model (<10 MB)
tiny = df_fp32_all[df_fp32_all["size_MB"] < 10].nlargest(1, "top1_%")
if not tiny.empty:
    row = tiny.iloc[0]
    print(f"\n💎 Best Tiny Model (<10 MB):")
    print(f"   {row['model']:40s} [{row['phase']}]")
    print(f"   Top-1: {row['top1_%']:.2f}%  |  Size: {row['size_MB']:.2f} MB  |  Efficiency: {row['top1_%'] / row['size_MB']:.2f} acc/MB")

# Best efficient (acc/MB)
best_eff = df_fp32_all.copy()
best_eff["eff"] = best_eff["top1_%"] / best_eff["size_MB"]
best_efficient = best_eff.nlargest(1, "eff").iloc[0]
print(f"\n⚡ Most Efficient (Accuracy per MB):")
print(f"   {best_efficient['model']:40s} [{best_efficient['phase']}]")
print(f"   Top-1: {best_efficient['top1_%']:.2f}%  |  Size: {best_efficient['size_MB']:.2f} MB  |  Efficiency: {best_efficient['eff']:.2f} acc/MB")

# Phase 4 summary
print(f"\n📊 Phase 4 Summary:")
print(f"   Models: 4 hybrid architectures")
print(f"   Best FP32: {phase4_fp32.nlargest(1, 'top1_%').iloc[0]['model']:35s} | {phase4_fp32['top1_%'].max():.2f}%")
phase4_int8 = phase4_comp[phase4_comp["precision"] == "INT8"]
if not phase4_int8.empty:
    print(f"   Best INT8: {phase4_int8.nlargest(1, 'top1_%').iloc[0]['model']:35s} | {phase4_int8['top1_%'].max():.2f}%")
    avg_drop = (phase4_fp32['top1_%'].mean() - phase4_int8['top1_%'].mean())
    print(f"   Avg QAT drop: {avg_drop:.2f}% (INT8 quantization robustness)")

print("\n" + "="*80)

## 6. Export Summary Tables

In [ ]:
# Phase 4 + Best models comparison export
summary_export = pd.DataFrame([
    {"category": "Phase 4 Best (FP32)", "model": best_overall["model"], "phase": best_overall["phase"],
     "top1_%": best_overall["top1_%"], "size_MB": best_overall["size_MB"], "params_M": best_overall["params_M"]},
])

if not tiny.empty:
    row = tiny.iloc[0]
    summary_export = pd.concat([
        summary_export,
        pd.DataFrame([{"category": "Best Tiny (<10MB)", "model": row["model"], "phase": row["phase"],
                      "top1_%": row["top1_%"], "size_MB": row["size_MB"], "params_M": row["params_M"]}])
    ], ignore_index=True)

summary_export = pd.concat([
    summary_export,
    pd.DataFrame([{"category": "Most Efficient", "model": best_efficient["model"], "phase": best_efficient["phase"],
                  "top1_%": best_efficient["top1_%"], "size_MB": best_efficient["size_MB"], "params_M": best_efficient["params_M"]}])
], ignore_index=True)

summary_export.to_csv(project_root / "results" / "phase4_best_models.csv", index=False)
print(f"✓ Exported phase4_best_models.csv")
display(summary_export)